# Overture Streets Walk Network Dataset for ArcGIS Pro
## Building a pedestrian Network Dataset from Overture Maps transportation data

This notebook produces a walking travel-mode Network Dataset for ArcGIS Pro Network Analyst
using road segment and connector (intersection node) data from the Overture Maps Foundation
`transportation` theme, downloaded by `OvertureStreets_Download.ipynb`.

The output Network Dataset includes:
- Walkable road segments filtered from the full Overture streets dataset
- WalkTime cost in minutes (5 km/h base, class-adjusted for steps and tracks)
- Overture connector points as explicit junction sources
- Network topology resolved from shared segment endpoints at connector locations

---

### Prerequisites

| Requirement | Notes |
|---|---|
| ArcGIS Pro 3.x | With **Network Analyst** extension licensed |
| `OvertureStreets_Download.ipynb` | Run first — populates `Segments` and `Connectors` in the GDB |
| `overture_walk_nd_template.xml` | Create once via ArcGIS Pro UI (see Step 5), then keep in this folder |

### Workflow Overview

| Step | Action |
|---|---|
| 1 | Verify environment and extensions |
| 2 | Verify input feature classes in the GDB |
| 3 | Filter segments to walkable road classes → `Overture_Walk_Segments` |
| 4 | Add `Length_M`, `Walk_Speed`, `WalkTime` fields |
| 5 | Create Network Dataset from template XML |
| 6 | Build Network Dataset |
| 7 | Verify and test |

In [1]:
import os

# =============================================================================
# CONFIGURATION — edit this section; all other cells run unchanged
# =============================================================================

# File Geodatabase written by OvertureStreets_Download.ipynb
GDB_PATH = r"F:\GIS\PROJECTS\Transportation\Streets Network\Streets_Network.gdb"
FDS_NAME = "OvertureStreets"

# Source feature class names (written by OvertureStreets_Download.ipynb)
FC_SEGMENTS_NAME   = "Segments"    # polyline — road segments
FC_CONNECTORS_NAME = "Connectors"  # point   — intersection nodes

# Output walk segments feature class name
# Must match the edge source name in the Network Dataset template XML
FC_WALK_NAME = "Overture_Walk_Segments"

# Network Dataset name — must match <Name> in overture_walk_nd_template.xml
ND_NAME = "Overture_Walk_ND"

# Network Dataset template XML — see Step 5 for creation instructions
# Keep in the same folder as this notebook
TEMPLATE_XML = os.path.join(os.getcwd(), "overture_walk_nd_template.xml")

# Walk model parameters
DEFAULT_WALK_KMH = 5.0   # base walk speed for roads and unclassified paths
MAX_SEGMENT_M    = 5000  # drop segments longer than this (metres)

# Study area clip extent (WGS84 decimal degrees) — Tahoe Basin
# Covers: South Lake Tahoe / Meyers → Truckee / Kings Beach
#         Homewood / Tahoe Pines (west) → Incline Village / Stateline (east)
BASIN_BBOX = {"xmin": -120.22, "ymin": 38.88, "xmax": -119.86, "ymax": 39.35}

# ---------------------------------------------------------------------------
# DERIVED PATHS — do not edit below this line
# ---------------------------------------------------------------------------
FDS_PATH           = os.path.join(GDB_PATH, FDS_NAME)
FC_SEGMENTS_PATH   = os.path.join(FDS_PATH, FC_SEGMENTS_NAME)
FC_CONNECTORS_PATH = os.path.join(FDS_PATH, FC_CONNECTORS_NAME)
FC_WALK_PATH       = os.path.join(FDS_PATH, FC_WALK_NAME)
ND_PATH            = os.path.join(FDS_PATH, ND_NAME)

print(f"GDB:        {GDB_PATH}")
print(f"FDS:        {FDS_NAME}")
print(f"Segments:   {FC_SEGMENTS_PATH}")
print(f"Connectors: {FC_CONNECTORS_PATH}")
print(f"Walk FC:    {FC_WALK_PATH}")
print(f"ND:         {ND_PATH}")
print(f"Template:   {TEMPLATE_XML}")
print(f"Basin bbox: {BASIN_BBOX}")

GDB:        F:\GIS\PROJECTS\Transportation\Streets Network\Streets_Network.gdb
FDS:        OvertureStreets
Segments:   F:\GIS\PROJECTS\Transportation\Streets Network\Streets_Network.gdb\OvertureStreets\Segments
Connectors: F:\GIS\PROJECTS\Transportation\Streets Network\Streets_Network.gdb\OvertureStreets\Connectors
Walk FC:    F:\GIS\PROJECTS\Transportation\Streets Network\Streets_Network.gdb\OvertureStreets\Overture_Walk_Segments
ND:         F:\GIS\PROJECTS\Transportation\Streets Network\Streets_Network.gdb\OvertureStreets\Overture_Walk_ND
Template:   c:\Users\amcclary\Documents\GitHub\Transportation\StreetNetwork\overture_walk_nd_template.xml
Basin bbox: {'xmin': -120.22, 'ymin': 38.88, 'xmax': -119.86, 'ymax': 39.35}


## Step 1: Verify Environment

Confirms ArcGIS Pro, arcpy, and the Network Analyst extension are available.
Run this cell before all others.

In [2]:
import sys
import importlib

print(f"Python: {sys.version}")
print()

checks = [
    ("arcpy",     "arcpy",     None),
    ("geopandas", "geopandas", "__version__"),
    ("pandas",    "pandas",    "__version__"),
]

all_ok = True
for name, module, attr in checks:
    try:
        m = importlib.import_module(module)
        ver = m.GetInstallInfo()["Version"] if module == "arcpy" else getattr(m, attr)
        print(f"  OK  {name:<12} {ver}")
    except ImportError:
        print(f"  MISSING  {name}")
        all_ok = False

print()

# Network Analyst extension check
try:
    import arcpy
    na_status = arcpy.CheckExtension("network")
    if na_status == "Available":
        arcpy.CheckOutExtension("network")
        print("  Network Analyst extension: checked out")
    else:
        print(f"  WARNING: Network Analyst extension status: {na_status}")
        all_ok = False
except Exception as e:
    print(f"  ERROR checking Network Analyst: {e}")
    all_ok = False

print()
print("Environment ready. Proceed to Step 2." if all_ok else "Resolve issues above before continuing.")

Python: 3.13.13 | packaged by Anaconda, Inc. | (main, Apr 22 2026, 16:49:16) [MSC v.1938 64 bit (AMD64)]

  OK  arcpy        3.7
  OK  geopandas    1.1.3
  OK  pandas       2.3.3

  Network Analyst extension: checked out

Environment ready. Proceed to Step 2.


c:\Users\amcclary\AppData\Local\ESRI\conda\envs\arcgispro-py3-plotly\Lib\site-packages\pyproj\network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


## Step 2: Verify Input Data

Checks that `OvertureStreets_Download.ipynb` has been run and the `Segments` and `Connectors`
feature classes exist in `Streets_Network.gdb`. Prints a road class breakdown so you can
confirm the data before filtering.

In [3]:
import arcpy
from collections import Counter

# ---------------------------------------------------------------------------
# CHECK EXISTENCE
# ---------------------------------------------------------------------------
existence_checks = [
    (GDB_PATH,           "File GDB"),
    (FDS_PATH,           "Feature Dataset (OvertureStreets)"),
    (FC_SEGMENTS_PATH,   "Segments FC"),
    (FC_CONNECTORS_PATH, "Connectors FC"),
]
all_exist = True
for path, label in existence_checks:
    exists = arcpy.Exists(path)
    status = "OK" if exists else "MISSING"
    if not exists:
        all_exist = False
    print(f"  {status:<8} {label}")

if not all_exist:
    raise RuntimeError(
        "One or more required inputs are missing. "
        "Run OvertureStreets_Download.ipynb (through cell 8c) first."
    )

print()

# ---------------------------------------------------------------------------
# FEATURE COUNTS
# ---------------------------------------------------------------------------
seg_count = int(arcpy.management.GetCount(FC_SEGMENTS_PATH)[0])
con_count = int(arcpy.management.GetCount(FC_CONNECTORS_PATH)[0])
print(f"Segments:   {seg_count:>8,}")
print(f"Connectors: {con_count:>8,}")

# ---------------------------------------------------------------------------
# ROAD CLASS BREAKDOWN
# ---------------------------------------------------------------------------
class_counts = Counter()
with arcpy.da.SearchCursor(FC_SEGMENTS_PATH, ["class"]) as cur:
    for row in cur:
        class_counts[row[0] or "(null)"] += 1

print("\nRoad class breakdown:")
for cls, cnt in sorted(class_counts.items(), key=lambda x: -x[1]):
    print(f"  {cls:<22} {cnt:>8,}")

  OK       File GDB
  OK       Feature Dataset (OvertureStreets)
  OK       Segments FC
  OK       Connectors FC

Segments:    103,278
Connectors:  154,828

Road class breakdown:
  service                  36,209
  residential              22,257
  footway                  16,272
  track                     9,761
  path                      5,319
  tertiary                  3,761
  secondary                 3,101
  unclassified              1,755
  primary                   1,651
  unknown                   1,117
  motorway                    708
  trunk                       524
  steps                       427
  cycleway                    328
  living_street                44
  pedestrian                   36
  bridleway                     8


## Step 3: Filter to Walkable Classes

**Why a whitelist instead of an exclude list?**

An exclude list leaves edge cases — private service roads, construction access tracks, and
unclassified rural lanes slip through. A strict whitelist (only classes where pedestrian
travel is reasonable) eliminates this problem. Same reasoning as the OSM workflow in
`OSM_Walk_Network_ND.ipynb`.

`motorway` and `trunk` are excluded: pedestrian access is generally prohibited on these
facility types.

The filtered result is written to `Overture_Walk_Segments` inside the same `OvertureStreets`
feature dataset. This is the edge source for the Network Dataset in Step 5.

**Note on the temp-name pattern:**
ArcGIS File GDB raises `Error 002851` if a feature class name is deleted and immediately
re-created. The workaround (write to a temp name, delete old copy, rename) is the same
pattern used in the OSM workflow.

In [4]:
import arcpy
import os

# ---------------------------------------------------------------------------
# WALKABLE ROAD CLASS WHITELIST
# ---------------------------------------------------------------------------
WALK_WHITELIST = {
    "primary",        "primary_link",
    "secondary",      "secondary_link",
    "tertiary",       "tertiary_link",
    "residential",
    "service",
    "living_street",
    "unclassified",
    "unknown",
    "footway",
    "pedestrian",
    "path",
    "cycleway",
    "track",
    "bridleway",
    "steps",
}

whitelist_sql  = ", ".join(f"'{c}'" for c in sorted(WALK_WHITELIST))
where_walkable = f'"class" IN ({whitelist_sql})'

# ---------------------------------------------------------------------------
# DELETE THE NETWORK DATASET FIRST
# arcpy.Exists is unreliable for NDs, so always attempt the delete.
# If it fails AND the ND still exists, ArcGIS Pro has a schema lock —
# remove Overture_Walk_ND and Walk_ServiceArea layers from the map first.
# ---------------------------------------------------------------------------
try:
    arcpy.management.Delete(ND_PATH)
    print(f"Deleted Network Dataset: {ND_NAME}  (Step 5 will rebuild from template)")
except Exception as e:
    if arcpy.Exists(ND_PATH):
        raise RuntimeError(
            f"Cannot delete {ND_NAME} — ArcGIS Pro likely has a schema lock.\n"
            "Remove the Overture_Walk_ND and Walk_ServiceArea layers from the Pro map "
            "(right-click → Remove), then re-run this cell.\n"
            f"Original error: {e}"
        )
    # ND is truly absent — spurious exception, continue

# ---------------------------------------------------------------------------
# TEMP NAMES — avoids Error 002851 on immediate FC rename
# ---------------------------------------------------------------------------
TEMP_FILTERED_NAME = FC_WALK_NAME + "_flt"
TEMP_FILTERED_PATH = os.path.join(FDS_PATH, TEMP_FILTERED_NAME)
TEMP_CLIPPED_NAME  = FC_WALK_NAME + "_clp"
TEMP_CLIPPED_PATH  = os.path.join(FDS_PATH, TEMP_CLIPPED_NAME)

for p in [TEMP_FILTERED_PATH, TEMP_CLIPPED_PATH]:
    if arcpy.Exists(p):
        arcpy.management.Delete(p)

# ---------------------------------------------------------------------------
# STEP 3A: FILTER TO WALKABLE CLASSES
# ---------------------------------------------------------------------------
total = int(arcpy.management.GetCount(FC_SEGMENTS_PATH)[0])
print(f"Source segments: {total:,}")
print("Applying whitelist filter...")

arcpy.conversion.FeatureClassToFeatureClass(
    in_features  = FC_SEGMENTS_PATH,
    out_path     = FDS_PATH,
    out_name     = TEMP_FILTERED_NAME,
    where_clause = where_walkable,
)
filtered_count = int(arcpy.management.GetCount(TEMP_FILTERED_PATH)[0])
print(f"  After whitelist: {filtered_count:,}  (removed {total - filtered_count:,})")

# ---------------------------------------------------------------------------
# STEP 3B: CLIP TO TAHOE BASIN BOUNDING BOX
# ---------------------------------------------------------------------------
print(f"Clipping to basin bbox {BASIN_BBOX}...")
sr = arcpy.SpatialReference(4326)
pts = arcpy.Array([
    arcpy.Point(BASIN_BBOX["xmin"], BASIN_BBOX["ymin"]),
    arcpy.Point(BASIN_BBOX["xmin"], BASIN_BBOX["ymax"]),
    arcpy.Point(BASIN_BBOX["xmax"], BASIN_BBOX["ymax"]),
    arcpy.Point(BASIN_BBOX["xmax"], BASIN_BBOX["ymin"]),
    arcpy.Point(BASIN_BBOX["xmin"], BASIN_BBOX["ymin"]),
])
bbox_fc = r"memory\tahoe_basin_bbox"
if arcpy.Exists(bbox_fc):
    arcpy.management.Delete(bbox_fc)
arcpy.management.CopyFeatures([arcpy.Polygon(pts, sr)], bbox_fc)

arcpy.analysis.Clip(TEMP_FILTERED_PATH, bbox_fc, TEMP_CLIPPED_PATH)
arcpy.management.Delete(TEMP_FILTERED_PATH)
arcpy.management.Delete(bbox_fc)

clipped_count = int(arcpy.management.GetCount(TEMP_CLIPPED_PATH)[0])
print(f"  After bbox clip: {clipped_count:,}  (removed {filtered_count - clipped_count:,})")

# ---------------------------------------------------------------------------
# REPLACE EXISTING WALK FC
# ---------------------------------------------------------------------------
if arcpy.Exists(FC_WALK_PATH):
    arcpy.management.Delete(FC_WALK_PATH)
    print(f"Deleted existing: {FC_WALK_NAME}")

arcpy.management.Copy(TEMP_CLIPPED_PATH, FC_WALK_PATH)
arcpy.management.Delete(TEMP_CLIPPED_PATH)

walk_count = int(arcpy.management.GetCount(FC_WALK_PATH)[0])
print(f"\nWalk segments (clipped to basin): {walk_count:,}  (removed {total - walk_count:,} total)")
print(f"Output: {FC_WALK_PATH}")

Deleted Network Dataset: Overture_Walk_ND  (Step 5 will rebuild from template)
Source segments: 103,278
Applying whitelist filter...
  After whitelist: 102,046  (removed 1,232)
Clipping to basin bbox {'xmin': -120.22, 'ymin': 38.88, 'xmax': -119.86, 'ymax': 39.35}...
  After bbox clip: 18,624  (removed 83,422)
Deleted existing: Overture_Walk_Segments

Walk segments (clipped to basin): 18,624  (removed 84,654 total)
Output: F:\GIS\PROJECTS\Transportation\Streets Network\Streets_Network.gdb\OvertureStreets\Overture_Walk_Segments


## Step 3b: Snap Walk Segment Endpoints to Each Other

The ND uses edge-only "Any Vertex" connectivity (no junction source). Two segments connect
only if they share a geometrically identical vertex. The XYTolerance in WGS84 is ~1 mm —
any endpoint gap larger than that breaks connectivity.

Snapping each `Overture_Walk_Segments` endpoint to the nearest other endpoint within 5 m
forces coincidence, restoring the street-graph connections the Overture connector IDs imply
but the GDB geometry may not reflect exactly after import and clip.

In [ ]:
import arcpy
import math
from collections import defaultdict

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------
SNAP_M     = 5.0    # snap tolerance in metres
LAT_CTR    = 39.1   # study area centre latitude for lon→m conversion
M_PER_LAT  = 111_320.0
M_PER_LON  = M_PER_LAT * math.cos(math.radians(LAT_CTR))
GRID_DEG   = (SNAP_M / min(M_PER_LAT, M_PER_LON)) * 4   # bucket width

def _cell(x, y):
    return int(x / GRID_DEG), int(y / GRID_DEG)

def _dist_m(x1, y1, x2, y2):
    return math.hypot((x2 - x1) * M_PER_LON, (y2 - y1) * M_PER_LAT)

# ---------------------------------------------------------------------------
# 1. READ ALL SEGMENT ENDPOINTS
# ---------------------------------------------------------------------------
endpoints = {}  # oid -> ((sx,sy), (ex,ey))
with arcpy.da.SearchCursor(FC_WALK_PATH, ["OID@", "SHAPE@"]) as cur:
    for oid, shp in cur:
        if shp:
            endpoints[oid] = (
                (shp.firstPoint.X, shp.firstPoint.Y),
                (shp.lastPoint.X,  shp.lastPoint.Y),
            )
print(f"Segments read: {len(endpoints):,}")

# ---------------------------------------------------------------------------
# 2. SPATIAL BUCKET INDEX
# ---------------------------------------------------------------------------
bucket = defaultdict(list)  # cell -> [(x,y,oid,slot)]
for oid, (s, e) in endpoints.items():
    for slot, (x, y) in enumerate((s, e)):
        ci, cj = _cell(x, y)
        for di in (-1, 0, 1):
            for dj in (-1, 0, 1):
                bucket[(ci + di, cj + dj)].append((x, y, oid, slot))

# ---------------------------------------------------------------------------
# 3. FIND SNAP TARGETS
# ---------------------------------------------------------------------------
snap      = {}  # (oid,slot) -> (tx,ty)
lone_ends = 0

for oid, (s, e) in endpoints.items():
    for slot, (x, y) in enumerate((s, e)):
        key = (oid, slot)
        if key in snap:
            continue
        ci, cj = _cell(x, y)
        best_d, best_xy = SNAP_M, None
        for bx, by, boid, _ in bucket[(ci, cj)]:
            if boid == oid:
                continue
            d = _dist_m(x, y, bx, by)
            if d < best_d:
                best_d, best_xy = d, (bx, by)
        if best_xy:
            snap[key] = best_xy
        else:
            lone_ends += 1

snapped_oids = {oid for (oid, _) in snap}
print(f"Endpoints within {SNAP_M} m of another: {len(snap):,}  |  lone (dead-end): {lone_ends:,}")
print(f"Segments to update: {len(snapped_oids):,}")

if not snap:
    print("\nNo gaps found — geometry is already coincident.")
    print("The connectivity problem is NOT a coordinate-gap issue; check WalkTime NULLs or ND build.")
else:
    # -----------------------------------------------------------------------
    # 4. APPLY VIA UpdateCursor + EDIT SESSION
    # -----------------------------------------------------------------------
    editor = arcpy.da.Editor(GDB_PATH)
    editor.startEditing(False, False)
    editor.startOperation()
    try:
        updated = 0
        with arcpy.da.UpdateCursor(FC_WALK_PATH, ["OID@", "SHAPE@"]) as cur:
            for row in cur:
                oid = row[0]
                if oid not in snapped_oids:
                    continue
                shp = row[1]
                if not shp or shp.partCount == 0:
                    continue
                new_parts = arcpy.Array()
                for p_idx in range(shp.partCount):
                    pts = [pt for pt in shp.getPart(p_idx) if pt is not None]
                    if not pts:
                        continue
                    if p_idx == 0 and (oid, 0) in snap:
                        sx, sy = snap[(oid, 0)]
                        pts[0] = arcpy.Point(sx, sy)
                    if p_idx == shp.partCount - 1 and (oid, 1) in snap:
                        ex, ey = snap[(oid, 1)]
                        pts[-1] = arcpy.Point(ex, ey)
                    new_parts.add(arcpy.Array(pts))
                row[1] = arcpy.Polyline(new_parts, shp.spatialReference)
                cur.updateRow(row)
                updated += 1
        editor.stopOperation()
        editor.stopEditing(True)
        print(f"Updated {updated:,} segments. Proceed to Steps 4 → 5.")
    except Exception:
        editor.stopOperation()
        editor.stopEditing(False)
        raise

## Step 4: Calculate WalkTime

Adds three fields to `Overture_Walk_Segments`:

| Field | Type | Description |
|---|---|---|
| `Length_M` | Double | Geodesic segment length in metres |
| `Walk_Speed` | Double | Walk speed in km/h based on road class |
| `WalkTime` | Double | Travel time in minutes — the Network Dataset cost attribute |

**Why geodesic length?**
The feature class is in WGS84 (geographic coordinates). `SHAPE@LENGTH` via a cursor returns
length in decimal degrees — meaningless for walk time. `CalculateGeometryAttributes` with
`LENGTH_GEODESIC` returns accurate metres regardless of spatial reference, no reprojection needed.

**Class-adjusted speed** applies reduced speeds to dedicated walking infrastructure (steps,
cycleways, tracks) and uses the default 5 km/h for all road types. Applying a surface
penalty to road centrelines would be incorrect — a pedestrian walking along a road shoulder
walks at normal pace regardless of that road's surface type.

> **Idempotent:** Re-running this cell recalculates all three fields without error,
> so you can safely rerun if you change `DEFAULT_WALK_KMH` or `MAX_SEGMENT_M`.

In [ ]:
import arcpy
import statistics

# ---------------------------------------------------------------------------
# WALK SPEED BY OVERTURE ROAD CLASS (km/h)
# ---------------------------------------------------------------------------
CLASS_SPEED = {
    "primary":        DEFAULT_WALK_KMH,
    "primary_link":   DEFAULT_WALK_KMH,
    "secondary":      DEFAULT_WALK_KMH,
    "secondary_link": DEFAULT_WALK_KMH,
    "tertiary":       DEFAULT_WALK_KMH,
    "tertiary_link":  DEFAULT_WALK_KMH,
    "residential":    DEFAULT_WALK_KMH,
    "service":        DEFAULT_WALK_KMH,
    "living_street":  DEFAULT_WALK_KMH,
    "unclassified":   DEFAULT_WALK_KMH,
    "unknown":        DEFAULT_WALK_KMH,
    "footway":        DEFAULT_WALK_KMH,
    "pedestrian":     DEFAULT_WALK_KMH,
    "path":           DEFAULT_WALK_KMH,
    "cycleway":       4.5,
    "track":          4.0,
    "bridleway":      4.0,
    "steps":          3.0,
}

# ---------------------------------------------------------------------------
# ADD FIELDS — idempotent, skips fields that already exist
# ---------------------------------------------------------------------------
existing = {f.name.lower() for f in arcpy.ListFields(FC_WALK_PATH)}
for fname, ftype in [("Length_M", "DOUBLE"), ("Walk_Speed", "DOUBLE"), ("WalkTime", "DOUBLE")]:
    if fname.lower() not in existing:
        arcpy.management.AddField(FC_WALK_PATH, fname, ftype, field_is_nullable="NULLABLE")
        print(f"Added field: {fname}")
    else:
        print(f"Field exists: {fname} (recalculating)")

# ---------------------------------------------------------------------------
# CALCULATE LENGTH, SPEED, WALKTIME — AND TRIM EXTREME SEGMENTS
# Single cursor pass: geodesic length via SHAPE@.getLength() avoids
# CalculateGeometryAttributes which fails on network-source FCs.
# Edit session required when FC participates in a network dataset.
# ---------------------------------------------------------------------------
print("\nCalculating Length_M, Walk_Speed, WalkTime...")
editor = arcpy.da.Editor(GDB_PATH)
editor.startEditing(False, False)   # no undo, non-versioned
editor.startOperation()
try:
    count_before = 0
    count_trimmed = 0
    with arcpy.da.UpdateCursor(
        FC_WALK_PATH,
        ["SHAPE@", "class", "Length_M", "Walk_Speed", "WalkTime"]
    ) as cur:
        for row in cur:
            count_before += 1
            geom       = row[0]
            length_m   = geom.getLength("GEODESIC", "METERS") if geom else 0.0
            road_class = str(row[1]).lower().strip() if row[1] else "unknown"
            if length_m < 0.5 or length_m > MAX_SEGMENT_M:
                cur.deleteRow()
                count_trimmed += 1
                continue
            speed  = CLASS_SPEED.get(road_class, DEFAULT_WALK_KMH)
            row[2] = length_m
            row[3] = speed
            row[4] = (length_m / 1000.0) / speed * 60.0
            cur.updateRow(row)

    editor.stopOperation()
    editor.stopEditing(True)   # save
except Exception:
    editor.stopOperation()
    editor.stopEditing(False)  # discard on error
    raise

count_after = count_before - count_trimmed
print(f"After length trim (0.5–{MAX_SEGMENT_M} m): {count_after:,}  (removed {count_trimmed:,})")

# ---------------------------------------------------------------------------
# SUMMARY STATS
# ---------------------------------------------------------------------------
walk_times = [
    r[0] for r in arcpy.da.SearchCursor(FC_WALK_PATH, ["WalkTime"])
    if r[0] is not None
]
print(f"\nWalkTime stats (minutes):")
print(f"  Count:  {len(walk_times):,}")
print(f"  Min:    {min(walk_times):.3f}")
print(f"  Median: {statistics.median(walk_times):.2f}")
print(f"  Max:    {max(walk_times):.2f}")

Added field: Length_M
Added field: Walk_Speed
Added field: WalkTime

Calculating Length_M, Walk_Speed, WalkTime...
After length trim (0.5–5000 m): 18,562  (removed 62)

WalkTime stats (minutes):
  Count:  18,562
  Min:    0.007
  Median: 1.06
  Max:    71.72


## Step 5: Create and Build Network Dataset

The Network Dataset is created from a template XML (`overture_walk_nd_template.xml`) that
encodes the edge source, junction source, connectivity policy, cost attributes, and travel
modes. **Creation only happens once** — on every subsequent run the existing ND structure
is reused and only the network topology is rebuilt against the current `Overture_Walk_Segments` data.

### Creating the Template (first run only)

1. In the **Catalog** pane, expand `Streets_Network.gdb` → right-click `OvertureStreets` → **New Network Dataset**
2. Name: `Overture_Walk_ND`
3. **Sources page** — add:
   - Edge source: `Overture_Walk_Segments` — connectivity: **Any Vertex**
   - Junction source: `Connectors` — connectivity: **End Point**
4. **Travel Attributes page** — add a cost attribute:
   - Name: `WalkTime` | Units: `Minutes` | Data type: `Double`
   - Evaluator → **Field** | Python expression: `!WalkTime!`
5. **Travel Modes page** — add a `Walking` mode:
   - Impedance: `WalkTime` | U-turns: **Allow U-turns Everywhere**
6. Finish the wizard and build the network
7. Right-click `Overture_Walk_ND` in the Catalog pane → **Export Network Dataset Template**
8. Save as `overture_walk_nd_template.xml` **in the same folder as this notebook**

### Subsequent runs (after updating Overture data)

Re-running Steps 3–4 updates `Overture_Walk_Segments` with new road data. Running Step 5
then rebuilds the network topology against that updated data — no recreation needed.

If you need to change the ND schema (add a travel mode, change an attribute), delete
`Overture_Walk_ND` manually in the ArcGIS Pro Catalog pane, then rerun this cell.

> **Source name must match:** The template records the exact feature class name used as
> the edge source. If you rename `Overture_Walk_Segments`, export a new template first.

In [ ]:
arcpy.env.workspace = FDS_PATH
print("FCs:", arcpy.ListFeatureClasses() or [])
print("Datasets:", arcpy.ListDatasets() or [])
arcpy.env.workspace = ""


FCs: ['Segments', 'Connectors', 'Overture_Walk_Segments_tmp', 'Overture_Walk_Segments']
Datasets: []


In [ ]:
import arcpy
import os

# ListDatasets with type="Network" does not reliably match NA Network Datasets
# in Pro 3.7 — use no type filter and match by name instead.
arcpy.env.workspace = FDS_PATH
existing_nds = [d for d in (arcpy.ListDatasets(ND_NAME) or []) if d == ND_NAME]
arcpy.env.workspace = ""
nd_exists = bool(existing_nds)
print(f"Network Dataset '{ND_NAME}' found: {nd_exists}")

if not nd_exists:
    # ----------------------------------------------------------------
    # GHOST CLEANUP
    # Even when arcpy.Exists / ListDatasets return False/empty, the GDB
    # catalog may still have the ND name registered. Attempt a direct
    # Delete on the ND path (works even when Exists returns False), then
    # remove any orphan junction FCs.
    # ----------------------------------------------------------------
    try:
        arcpy.management.Delete(ND_PATH)
        print(f"  Cleared ghost registration: {ND_NAME}")
    except Exception:
        pass  # expected when ND is truly absent

    arcpy.env.workspace = FDS_PATH
    orphan_fcs = arcpy.ListFeatureClasses(ND_NAME + "*") or []
    arcpy.env.workspace = ""
    for name in orphan_fcs:
        arcpy.management.Delete(os.path.join(FDS_PATH, name))
        print(f"  Removed orphan FC: {name}")

    for suffix in ("_old", "_tmp"):
        old_path = os.path.join(FDS_PATH, ND_NAME + suffix)
        if arcpy.Exists(old_path):
            arcpy.management.Delete(old_path)
            print(f"  Removed: {ND_NAME + suffix}")

    if not os.path.exists(TEMPLATE_XML):
        raise FileNotFoundError(
            f"Template not found: {TEMPLATE_XML}\n"
            "See Step 5 markdown for creation instructions."
        )

    print(f"Template: {TEMPLATE_XML}")
    print("Creating Network Dataset from template...")
    arcpy.na.CreateNetworkDatasetFromTemplate(
        network_dataset_template = TEMPLATE_XML,
        output_feature_dataset   = FDS_PATH,
    )
    print(f"  Created: {ND_PATH}")
else:
    print("  Rebuilding against current Overture_Walk_Segments data.")

print("\nBuilding network — expect 1-5 minutes for Tahoe-area data...")
arcpy.na.BuildNetwork(ND_PATH)
print("Build complete.")

Network Dataset 'Overture_Walk_ND' found: True
  Rebuilding against current Overture_Walk_Segments data.

Building network — expect 1-5 minutes for Tahoe-area data...
Build complete.


## Step 6: Verify Network Dataset

Confirms the Network Dataset built successfully and reports source information.
After running this cell, open ArcGIS Pro and run a quick Service Area solve to confirm
routing produces sensible pedestrian isochrones.

In [ ]:
import arcpy

desc       = arcpy.Describe(ND_PATH)
walk_count = int(arcpy.management.GetCount(FC_WALK_PATH)[0])
con_count  = int(arcpy.management.GetCount(FC_CONNECTORS_PATH)[0])

print(f"Network Dataset: {desc.name}")
print(f"  Location:         {ND_PATH}")
print(f"  Edge sources:     {[s.name for s in desc.edgeSources]}")
print(f"  Junction sources: {[s.name for s in desc.junctionSources]}")
print(f"  Walk segments:    {walk_count:,}")
print(f"  Connectors:       {con_count:,}")

print("\nNetwork Analyst ready.")
print("Quick test: Analysis tab > Service Area | Travel Mode: Walking | Cutoffs: 5 10 15")
print("Expected:   Walk isochrones covering Tahoe basin / Truckee / Carson City")

Network Dataset: Overture_Walk_ND
  Location:         F:\GIS\PROJECTS\Transportation\Streets Network\Streets_Network.gdb\OvertureStreets\Overture_Walk_ND
  Edge sources:     ['Overture_Walk_Segments']
  Junction sources: ['Overture_Walk_ND_Junctions']
  Walk segments:    18,562
  Connectors:       154,828

Network Analyst ready.
Quick test: Analysis tab > Service Area | Travel Mode: Walking | Cutoffs: 5 10 15
Expected:   Walk isochrones covering Tahoe basin / Truckee / Carson City


## Known Limitations

### Overture Connector Topology
Overture segments reference connector IDs to encode intersection topology. This notebook
relies on ArcGIS Network Analyst's planar intersection detection at build time. In areas
where segment endpoints do not perfectly snap to connector point locations in the GDB,
some intersection connections may be missed. For highest-quality topology, snap segment
endpoints to their referenced connector points before building the Network Dataset.

### WalkTime Assumes Flat Grade
The 5 km/h model does not account for slope. Tahoe's steep terrain means actual walk times
will be longer than modelled on uphill segments, shorter on downhill segments. For a more
accurate model, use a slope-adjusted speed function with a DEM.

### motorway and trunk Excluded
`motorway` and `trunk` class segments are excluded from the walk network. If your study
area includes pedestrian-accessible shared-use paths alongside these roads, add those
classes back to `WALK_WHITELIST` in Step 3.

### Overture Data Currency
The network reflects the Overture release downloaded by `OvertureStreets_Download.ipynb`.
Re-run that notebook (to download a newer release and repopulate `Segments`/`Connectors`)
then re-run this notebook to update the walk network.

---

## Data Source

| Source | Licence |
|---|---|
| Overture Maps Foundation | CDLA Permissive 2.0 |

> When publishing analysis based on this network, acknowledge the Overture Maps Foundation
> and its data contributors as the source.

## Step 7: Import Facility Points and Run Walk Service Area

Downloads a point feature layer from TRPA's Transportation Planning map service,
imports it to the geodatabase, then runs a 5 / 10 / 20-minute walk Service Area
solve against `Walk_Network`.

In [ ]:
import requests
import json
import tempfile
import arcpy
import os

# ---------------------------------------------------------------------------
# REST ENDPOINT — TRPA Transportation Planning MapServer layer 16
# ---------------------------------------------------------------------------
REST_URL = "https://maps.trpa.org/server/rest/services/Transportation_Planning/MapServer/16"

# Get layer metadata so we can name the output FC after the actual layer name
meta       = requests.get(REST_URL, params={"f": "json"}).json()
layer_name = meta.get("name", "Transportation_Points").replace(" ", "_")
geom_type  = meta.get("geometryType", "unknown")
print(f"Layer:    {layer_name}")
print(f"Geometry: {geom_type}")
print(f"Fields:   {[f['name'] for f in meta.get('fields', [])][:10]}")

# ---------------------------------------------------------------------------
# QUERY ALL FEATURES
# ---------------------------------------------------------------------------
resp = requests.get(
    REST_URL + "/query",
    params={
        "where":          "1=1",
        "outFields":      "*",
        "returnGeometry": "true",
        "outSR":          "4326",
        "f":              "json",
    },
)
resp.raise_for_status()
data = resp.json()

if data.get("exceededTransferLimit"):
    print("WARNING: server hit transfer limit — results may be incomplete.")

print(f"Features received: {len(data.get('features', [])):,}")

# ---------------------------------------------------------------------------
# IMPORT TO GDB
# ---------------------------------------------------------------------------
tmp_json  = os.path.join(tempfile.gettempdir(), f"{layer_name}.json")
POINTS_FC = os.path.join(GDB_PATH, layer_name)

with open(tmp_json, "w") as f:
    json.dump(data, f)

if arcpy.Exists(POINTS_FC):
    arcpy.management.Delete(POINTS_FC)
arcpy.conversion.JSONToFeatures(tmp_json, POINTS_FC)

imported = int(arcpy.management.GetCount(POINTS_FC)[0])
print(f"Imported {imported:,} features -> {POINTS_FC}")

Layer:    School
Geometry: esriGeometryPoint
Fields:   ['OBJECTID', 'NAME', 'TYPE', 'COUNTY', 'CELLNAME', 'ENROLLMENT', 'FRPM', 'FRPM_DENS', 'ADDRESS', 'ENROLL_TEX']
Features received: 21
Imported 21 features -> F:\GIS\PROJECTS\Transportation\Streets Network\Streets_Network.gdb\School


## Step 7: Run Walk Service Area

The Network Analyst solve must be run outside Jupyter due to a conflict between the
arcpy NA C extension and the Jupyter kernel.

**Run `run_walk_service_area.py`** from the ArcGIS Pro Python window:

```python
exec(open(r"C:\Users\amcclary\Documents\GitHub\Transportation\StreetNetwork\run_walk_service_area.py").read())
```

Output: `Walk_ServiceArea_5_10_20min` in `Streets_Network.gdb` (5, 10, and 20-minute walk polygons for each school).